# miniLLM L1 — полное обучение на Kaggle GPU

Этот блокнот запускает **первый настоящий 20M scaling checkpoint**, а не обучение
финального ассистента. По умолчанию обучается attention-кандидат на всех 31.09M токенах
пилотного RU/EN-корпуса. Edge-контроль можно включить одной строкой.

Перед запуском:

1. В Kaggle откройте **Settings → Accelerator → GPU**.
2. Включите **Internet**: код и готовый 48 MB data bundle загружаются одним git clone.
3. Нажмите **Run All**. `DATA_INPUT` настраивать не нужно.
4. После обучения выберите **Save Version**, чтобы checkpoints из `/kaggle/working`
   сохранились как output текущей версии.

Ожидаемый результат пока — слабая base model и измерения обучения. 31M токенов
недостаточно для готового локального ассистента.


In [ ]:
# ===================== НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ =====================
from pathlib import Path

REPOSITORY = "https://github.com/ovococjsjs-gif/miniLLM.git"
BRANCH = "arena/01a01c73-minillm"
# Точная проверенная ревизия training workflow — не меняйте для первого run.
REVISION = "fb8a3f19a22ec017838256a093a4aa47e0dfb846"

# По умолчанию обучаем quality-control. Чтобы обучить обе руки последовательно:
# VARIANTS = ["attention", "edge"]
VARIANTS = ["attention"]

# Необязательно: путь к распакованному репозиторию в /kaggle/input вместо git clone.
REPO_INPUT = ""
# Обычно оставьте пустым: готовый 48 MB архив уже лежит рядом с notebook в GitHub
# и будет получен при git clone. Необязательно можно указать свой Kaggle Dataset.
DATA_INPUT = ""

# Для продолжения старого Kaggle run укажите его read-only output directory.
# Notebook скопирует его в /kaggle/working и продолжит с последнего step-*.pt.
RESUME_RUNS = {
    "attention": "",
    "edge": "",
}

BATCH_SIZE = 8
SEQUENCE_LENGTH = 512
GRADIENT_ACCUMULATION = 8
CHECKPOINT_INTERVAL = 250
EVAL_INTERVAL = 100
EVAL_BATCHES = 20
SEED = 42
FORCE_RESTART = False

WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "miniLLM"
DATA_BUILD_ROOT = WORKING / "minillm-l1-data"
RUN_ROOT = WORKING / "minillm-runs"
EXPORT_ROOT = WORKING / "minillm-export"

assert set(VARIANTS) <= {"attention", "edge"} and VARIANTS
print("Будут обучены:", VARIANTS)


## 1. Получение кода

Ячейка фиксирует фактический Git commit в каждом checkpoint. Установка выполняется без
замены Kaggle-версии PyTorch.


In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys

if REPO_DIR.exists() and not (REPO_DIR / "pyproject.toml").exists():
    raise RuntimeError(f"Неполный репозиторий уже существует: {REPO_DIR}")

if not REPO_DIR.exists():
    if REPO_INPUT:
        source = Path(REPO_INPUT)
        if not (source / "pyproject.toml").exists():
            matches = list(source.rglob("pyproject.toml"))
            if len(matches) != 1:
                raise RuntimeError("Не удалось однозначно найти miniLLM в REPO_INPUT")
            source = matches[0].parent
        shutil.copytree(source, REPO_DIR)
    else:
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, REPOSITORY, str(REPO_DIR)],
            check=True,
        )

os.chdir(REPO_DIR)
source_snapshot_created = False
if not (REPO_DIR / ".git").exists():
    # Kaggle source datasets often omit .git. Create a clean content-addressed snapshot
    # so trainer provenance and dirty-worktree protection still work.
    subprocess.run(["git", "init", "-q"], check=True)
    subprocess.run(["git", "config", "user.name", "Kaggle miniLLM"], check=True)
    subprocess.run(["git", "config", "user.email", "kaggle@localhost"], check=True)
    subprocess.run(["git", "add", "."], check=True)
    subprocess.run(["git", "commit", "-q", "-m", "Kaggle source snapshot"], check=True)
    source_snapshot_created = True
if REVISION and not REVISION.startswith("__") and not source_snapshot_created:
    subprocess.run(["git", "checkout", "--detach", REVISION], check=True)

for package, requirement in [("tokenizers", "tokenizers>=0.20")]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", requirement], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"],
    check=True,
)

GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Код готов. Commit:", GIT_COMMIT)


## 2. Проверка GPU и выбор точности

Notebook сначала проверяет, содержит ли PyTorch kernels для выданной карты. Для Kaggle P100
автоматически устанавливается официальный PyTorch CUDA 12.6 с поддержкой `sm_60`; Ampere
и более новые GPU используют BF16, а T4/P100 — FP16. Реальная CUDA-операция и AdamW
smoke-test выполняются до обучения.


In [ ]:
import json
import platform

# Kaggle can assign a Pascal P100 (sm_60) while its default CUDA 12.8/13 PyTorch wheel
# contains kernels only for sm_70+. Query in child processes so replacing torch does not
# require restarting this notebook kernel.
capability_text = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
    text=True,
).splitlines()[0].strip()
major, minor = (int(part) for part in capability_text.split(".")[:2])
required_arch = f"sm_{major}{minor}"

arch_query = (
    "import json, torch; "
    "print('MINILLM_ARCH=' + json.dumps(torch.cuda.get_arch_list()))"
)
def installed_architectures() -> list[str]:
    completed = subprocess.run(
        [sys.executable, "-c", arch_query],
        text=True,
        capture_output=True,
        check=True,
    )
    marker = [line for line in completed.stdout.splitlines() if line.startswith("MINILLM_ARCH=")]
    if len(marker) != 1:
        raise RuntimeError("Не удалось определить CUDA architectures установленного PyTorch")
    return json.loads(marker[0].split("=", 1)[1])

architectures = installed_architectures()
if required_arch not in architectures:
    if major <= 6:
        print(
            f"PyTorch {required_arch} не поддерживает. Устанавливается официальный "
            "PyTorch 2.7.1 CUDA 12.6 с поддержкой Pascal/P100..."
        )
        subprocess.run(
            [
                sys.executable, "-m", "pip", "install",
                "--no-cache-dir", "--upgrade", "--force-reinstall",
                "torch==2.7.1",
                "--index-url", "https://download.pytorch.org/whl/cu126",
            ],
            check=True,
        )
        architectures = installed_architectures()
    if required_arch not in architectures:
        raise RuntimeError(
            f"Установленный PyTorch не содержит {required_arch}: {architectures}"
        )

# Real allocation and optimizer step catch exactly the no-kernel-image failure before training.
probe_source = r"""
import json
import platform
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable')
capability = torch.cuda.get_device_capability(0)
tensor = torch.ones(1024, device='cuda')
value = tensor.square().sum()
torch.cuda.synchronize()
fused = True
try:
    parameter = torch.nn.Parameter(torch.ones(8, device='cuda'))
    optimizer = torch.optim.AdamW([parameter], fused=True)
    parameter.square().sum().backward()
    optimizer.step()
    torch.cuda.synchronize()
except Exception:
    fused = False
precision = 'bf16' if capability[0] >= 8 else 'fp16'
result = {
    'name': torch.cuda.get_device_name(0),
    'compute_capability': list(capability),
    'supported_architectures': torch.cuda.get_arch_list(),
    'memory_gib': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    'precision': precision,
    'fused_adamw': fused,
    'torch': torch.__version__,
    'torch_cuda': torch.version.cuda,
    'python': platform.python_version(),
    'cuda_smoke_value': float(value),
}
print('MINILLM_GPU=' + json.dumps(result))
"""
probe = subprocess.run(
    [sys.executable, "-c", probe_source],
    text=True,
    capture_output=True,
)
if probe.returncode:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError("CUDA kernel smoke-test не прошёл")
marker = [line for line in probe.stdout.splitlines() if line.startswith("MINILLM_GPU=")]
if len(marker) != 1:
    raise RuntimeError("CUDA probe не вернул отчёт")
GPU_INFO = json.loads(marker[0].split("=", 1)[1])
PRECISION = GPU_INFO["precision"]
FUSED_OPTIMIZER = GPU_INFO["fused_adamw"]
print(json.dumps(GPU_INFO, indent=2))


## 3. Данные

Готовый `l1-github-pilot-data-v1.tar.gz` лежит в папке `kaggle/` того же GitHub
репозитория. Notebook найдёт его после clone, распакует 48 MB bundle и проверит полные
SHA-256. Ничего добавлять через **Add Input** не нужно.

Только если bundle отсутствует, notebook скачает три источника на точных commit SHA,
применит policy/PII/deduplication, упакует токены и удалит промежуточные копии. GitHub
rebuild заметно дольше и может зависеть от скорости Kaggle Internet.


In [ ]:
import importlib.util
import json
import tarfile

prepare_path = REPO_DIR / "scripts" / "prepare_l1_data.py"
spec = importlib.util.spec_from_file_location("prepare_l1_data", prepare_path)
prepare_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prepare_module)
BUNDLE_NAME = "l1-github-pilot-data-v1.tar.gz"

def token_directories(root: Path) -> list[Path]:
    candidates = []
    if not root.exists() or root.is_file():
        return candidates
    for train in root.rglob("train.bin"):
        parent = train.parent
        required = [
            parent / "validation.bin",
            parent / "test.bin",
            parent / "manifest.json",
            parent / "train.bin.json",
            parent / "validation.bin.json",
            parent / "test.bin.json",
        ]
        if all(path.exists() for path in required):
            candidates.append(parent)
    return candidates

def locate_tokens(root: Path) -> Path:
    candidates = token_directories(root)
    if len(candidates) != 1:
        raise RuntimeError(f"Ожидался один token-stream в {root}, найдено {len(candidates)}")
    return candidates[0]

def tokens_from_input(source: Path) -> Path:
    if source.is_dir():
        ready = token_directories(source)
        if len(ready) == 1:
            return ready[0]
        archives = list(source.rglob(BUNDLE_NAME))
    else:
        archives = [source] if source.name == BUNDLE_NAME else []
    if len(archives) != 1:
        raise RuntimeError(
            f"В {source} нет одного готового token-stream или {BUNDLE_NAME}"
        )
    destination = DATA_BUILD_ROOT / "prepacked"
    if destination.exists() and len(token_directories(destination)) != 1:
        shutil.rmtree(destination)
    if not destination.exists():
        destination.mkdir(parents=True)
        print(f"Распаковка готового 48 MB bundle: {archives[0]}")
        with tarfile.open(archives[0], "r:gz") as archive:
            archive.extractall(destination, filter="data")
    return locate_tokens(destination)

input_source = Path(DATA_INPUT) if DATA_INPUT else None
if input_source is None:
    automatic_bundles = []
    for search_root in (REPO_DIR / "kaggle", Path("/kaggle/input")):
        if search_root.exists():
            automatic_bundles.extend(search_root.rglob(BUNDLE_NAME))
    automatic_bundles = sorted(set(automatic_bundles))
    if len(automatic_bundles) == 1:
        input_source = automatic_bundles[0]
        print("Автоматически найден data bundle:", input_source)
    elif len(automatic_bundles) > 1:
        raise RuntimeError("Найдено несколько L1 bundles; укажите нужный в DATA_INPUT")

if input_source is not None:
    TOKENS_DIR = tokens_from_input(input_source)
    DATA_IDENTITY = prepare_module.verify_token_streams(TOKENS_DIR)
else:
    print("Готовый bundle не найден; начинается медленная сборка из GitHub sources.")
    subprocess.run(
        [
            sys.executable,
            str(prepare_path),
            "--root", str(DATA_BUILD_ROOT),
            "--cleanup-intermediates",
        ],
        cwd=REPO_DIR,
        check=True,
    )
    TOKENS_DIR = DATA_BUILD_ROOT / "tokens"
    DATA_IDENTITY = prepare_module.verify_token_streams(TOKENS_DIR)

print("Данные проверены:", TOKENS_DIR)
print(json.dumps(DATA_IDENTITY, indent=2))


## 4. Проверка плана до расходования GPU-времени

Ожидается 949 optimizer steps и примерно 31.10M фактически обработанных токенов на каждый
вариант. Dry-run ничего не обучает.


In [ ]:
MODEL_CONFIGS = {
    "attention": "configs/l1_attention_20m.json",
    "edge": "configs/l1_edge_20m.json",
}

def common_train_arguments(variant: str, output: Path) -> list[str]:
    return [
        sys.executable,
        "scripts/train_l1.py",
        "--model", MODEL_CONFIGS[variant],
        "--tokens", str(TOKENS_DIR),
        "--output", str(output),
        "--device", "cuda",
        "--precision", PRECISION,
        "--batch-size", str(BATCH_SIZE),
        "--sequence-length", str(SEQUENCE_LENGTH),
        "--gradient-accumulation", str(GRADIENT_ACCUMULATION),
        "--checkpoint-interval", str(CHECKPOINT_INTERVAL),
        "--eval-interval", str(EVAL_INTERVAL),
        "--eval-batches", str(EVAL_BATCHES),
        "--seed", str(SEED),
        "--fused-optimizer" if FUSED_OPTIMIZER else "--no-fused-optimizer",
    ]

PLANS = {}
for variant in VARIANTS:
    command = common_train_arguments(variant, RUN_ROOT / variant) + ["--dry-run"]
    PLANS[variant] = json.loads(subprocess.check_output(command, cwd=REPO_DIR, text=True))
    print(variant, json.dumps(PLANS[variant], indent=2))
    assert PLANS[variant]["steps"] == 949
    assert PLANS[variant]["target_tokens"] == 31_094_503


## 5. Обучение

Это долгая ячейка. Не останавливайте её только потому, что несколько минут нет нового
текста: trainer пишет одну JSON-строку на шаг в `metrics.jsonl`, а notebook ждёт завершения
процесса.

При повторном запуске ячейка автоматически продолжает существующий run с последнего
`step-*.pt`. Для переноса между Kaggle sessions добавьте output предыдущей версии как
Dataset и укажите каталог run в `RESUME_RUNS`. Параметры batch/precision при resume должны
совпадать.


In [ ]:
import re

def latest_checkpoint(directory: Path) -> Path | None:
    checkpoints = []
    for path in directory.glob("step-*.pt"):
        match = re.fullmatch(r"step-(\d+)\.pt", path.name)
        if match:
            checkpoints.append((int(match.group(1)), path))
    return max(checkpoints, default=(0, None))[1]

RUN_ROOT.mkdir(parents=True, exist_ok=True)
COMPLETED_RUNS = {}
for variant in VARIANTS:
    output = RUN_ROOT / variant
    previous = RESUME_RUNS.get(variant, "")

    if FORCE_RESTART and output.exists():
        shutil.rmtree(output)
    if previous and not output.exists():
        print(f"Копирование предыдущего {variant} run в writable storage...")
        shutil.copytree(Path(previous), output)

    if (output / "l1-summary.json").exists():
        print(f"{variant}: полный run уже завершён, обучение пропущено")
        COMPLETED_RUNS[variant] = output
        continue

    resume = latest_checkpoint(output) if output.exists() else None
    if output.exists() and resume is None:
        metrics = output / "metrics.jsonl"
        files = set(output.iterdir())
        failed_before_first_step = (
            not files or (files == {metrics} and metrics.stat().st_size == 0)
        )
        if failed_before_first_step:
            print(f"{variant}: удаление пустого run после неудачного CUDA startup")
            shutil.rmtree(output)
        else:
            raise RuntimeError(
                f"{output} существует, но resume checkpoint отсутствует. "
                "Проверьте каталог или установите FORCE_RESTART=True."
            )

    command = common_train_arguments(variant, output)
    if resume is not None:
        print(f"{variant}: продолжение с {resume.name}")
        command += ["--resume", str(resume)]
    else:
        print(f"{variant}: новое обучение")

    subprocess.run(command, cwd=REPO_DIR, check=True)
    if not (output / "l1-summary.json").exists():
        raise RuntimeError(f"{variant}: trainer завершился без итогового отчёта")
    COMPLETED_RUNS[variant] = output

print("Завершены:", sorted(COMPLETED_RUNS))


## 6. Графики и итоговые метрики


In [ ]:
import matplotlib.pyplot as plt

for variant, directory in COMPLETED_RUNS.items():
    records = [
        json.loads(line)
        for line in (directory / "metrics.jsonl").read_text().splitlines()
        if line.strip()
    ]
    steps = [row["step"] for row in records]
    losses = [row["train_main_loss"] for row in records]
    validation = [
        (row["step"], row["validation_main_loss"])
        for row in records
        if "validation_main_loss" in row
    ]
    summary = json.loads((directory / "l1-summary.json").read_text())
    print("\n", variant, json.dumps(summary["summary"], indent=2))

    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses, alpha=0.55, label="train main loss")
    if validation:
        plt.plot(*zip(*validation), marker="o", label="validation main loss")
    plt.title(f"L1 {variant}")
    plt.xlabel("optimizer step")
    plt.ylabel("loss")
    plt.grid(alpha=0.2)
    plt.legend()
    plt.show()


## 7. Детерминированная RU/EN completion-диагностика

Нулевой или низкий score на 31M токенах не считается поломкой. Нам важно увидеть, исчез ли
newline/repetition collapse и появился ли связный текст.


In [ ]:
for variant, directory in COMPLETED_RUNS.items():
    completion_output = directory / "completion-smoke.json"
    subprocess.run(
        [
            sys.executable,
            "scripts/evaluate_completions.py",
            str(directory / "best-inference.pt"),
            "--tokenizer", "artifacts/tokenizer-github-pilot-v1/tokenizer.json",
            "--output", str(completion_output),
            "--device", "cuda",
        ],
        cwd=REPO_DIR,
        check=True,
    )
    report = json.loads(completion_output.read_text())
    print(f"\n{variant}: {report['passed']}/{report['total']}")
    for case in report["cases"]:
        print(case["id"], "=>", repr(case["completion"]))


## 8. Компактный экспорт

Полные каталоги `minillm-runs` содержат optimizer checkpoints и нужны для resume.
`minillm-export` содержит только лучший inference checkpoint, логи, конфигурацию,
токенизатор и provenance — его удобнее скачать для анализа.


In [ ]:
import datetime

EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
run_manifest = {
    "schema_version": 1,
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "git_commit": GIT_COMMIT,
    "gpu": GPU_INFO,
    "data": DATA_IDENTITY,
    "variants": {},
}
for variant, directory in COMPLETED_RUNS.items():
    destination = EXPORT_ROOT / variant
    destination.mkdir(parents=True, exist_ok=True)
    names = [
        "best-inference.pt",
        "l1-summary.json",
        "summary.json",
        "metrics.jsonl",
        "completion-smoke.json",
    ]
    for name in names:
        source = directory / name
        if source.exists():
            shutil.copy2(source, destination / name)
    shutil.copy2(REPO_DIR / MODEL_CONFIGS[variant], destination / "model-config.json")
    run_manifest["variants"][variant] = {
        "run_directory": str(directory),
        "export_directory": str(destination),
        "plan": PLANS[variant],
    }

shutil.copy2(
    REPO_DIR / "artifacts/tokenizer-github-pilot-v1/tokenizer.json",
    EXPORT_ROOT / "tokenizer.json",
)
shutil.copy2(
    REPO_DIR / "artifacts/tokenizer-github-pilot-v1/manifest.json",
    EXPORT_ROOT / "tokenizer-manifest.json",
)
(EXPORT_ROOT / "kaggle-run-manifest.json").write_text(
    json.dumps(run_manifest, indent=2) + "\n"
)

print("Полные resume checkpoints:", RUN_ROOT)
print("Компактный экспорт:", EXPORT_ROOT)
print("Теперь нажмите Kaggle Save Version, чтобы сохранить /kaggle/working.")


## После завершения

Сохраните ссылку на Kaggle version и не удаляйте:

- `minillm-runs/<variant>/best.pt` и `step-*.pt` — для проверки/resume;
- `minillm-export/<variant>/best-inference.pt` — для генерации;
- `metrics.jsonl`, `l1-summary.json`, `kaggle-run-manifest.json` — для анализа.

Чтобы продолжить run в новой сессии, добавьте output старой версии через **Add Input** и
впишите путь к каталогу соответствующего варианта в `RESUME_RUNS`. Не меняйте precision,
batch size, sequence length или accumulation при resume.
